<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>GraphRAG Data Analysis </h3>
    The following is an overview of the supported features:
    <ul>
    <li>Generates a summarization report of the codebase</li>
    <li>Generates a visualization report of the codebase</li>
    <li>Can query the database with specific inquiries about the codebase</li>
    </ul>
</div>

In [ ]:
##############################################################################
# Imports
##############################################################################
from utils.graphrag_utils import DependencyAnalyzer
from pipelines.base.analysis import run_full_pipeline
# from utils.visualization_utils import visualize_dependencies
from dotenv import load_dotenv
import logging
import os
from IPython.display import display, Markdown

logging.basicConfig(level=os.environ.get('LOGLEVEL', 'INFO').upper())

load_dotenv()

import asyncio
import nest_asyncio

nest_asyncio.apply()

In [ ]:
#########################################################################
# MlFlow
#########################################################################
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

### GraphRAG Queries

In [ ]:
##############################################################################
# Instance Variables
##############################################################################
from utils import code_utils

_GRAPHRAG_SOURCE_PATH = f"{os.path.abspath('graph_rag_app')}/source"
_GIT_REPO = os.getenv("GIT_REPO", "")
_GIT_BRANCH = os.getenv("GIT_BRANCH", "")
_GIT_SLUG = os.getenv("GIT_SLUG") or (code_utils.generate_slug_from_repo(_GIT_REPO, _GIT_BRANCH) if _GIT_REPO else "")
_MULTI_REPO = os.getenv("MULTI_REPO", "false").lower() == "true"
analyzer = DependencyAnalyzer(_GRAPHRAG_SOURCE_PATH, git_slug=_GIT_SLUG, multi_repo=_MULTI_REPO)

#### Adhoc Queries (Perform ad-hoc queries against the GraphRAG index)

In [ ]:
##############################################################################
# Sample Questions:
# 1. Which modules or components would be riskiest to refactor first? Include the fully qualified names.
# 2. What migration order would be recommended when refactoring to reduce breaking changes? Include the fully qualified names.
# 3. Which modules or components would be the least risky to migrate first? Include the fully qualified names.
# 4. Are there any vulnerable dependencies or libraries? Include the fully qualified names.
# 5. (For multiple repos) In what order should these git repositories be refactored, from least risky to most risky? Include all the git repositories you can find and their git repository links.
# 6. (For multiple repos) What categories of applications are available in this codebase? Include all the git repositories you can find and their git repository links.
##############################################################################
question = """
Which modules or components would be riskiest to refactor first?
Include the fully qualified names.
"""
response = await analyzer.query_with_llm(question)
display(Markdown(response))

#### Generate a migration report

In [ ]:
pipeline_result = run_full_pipeline(_GRAPHRAG_SOURCE_PATH, git_repo=_GIT_REPO, git_branch=_GIT_BRANCH, multi_repo=_MULTI_REPO)

display(Markdown(pipeline_result))

#### Generate a visualization report

In [ ]:
# visualize_dependencies(pipeline_result)